In [1]:
# TAHAP 1A IMPORT LIBRARY

print("Mulai mengimpor library yang dibutuhkan...")

import requests
import pandas as pd
from io import StringIO

print("Library berhasil diimpor.")

Mulai mengimpor library yang dibutuhkan...
Library berhasil diimpor.


In [2]:
# TAHAP 1B KONFIGURASI API E-STAT

print("Mulai melakukan konfigurasi API e-Stat...")

# APP ID API e-Stat
APP_ID = "69fbbb74fa35bfa86664ac12004c7b8b7296ea15"

# ID dataset pengeluaran rumah tangga
STATS_DATA_ID = "0000010212"

print("Konfigurasi API berhasil disiapkan.")
print(f"Stats Data ID : {STATS_DATA_ID}")

Mulai melakukan konfigurasi API e-Stat...
Konfigurasi API berhasil disiapkan.
Stats Data ID : 0000010212


In [3]:
# TAHAP 1C PEMBENTUKAN URL REQUEST API

print("Mulai membentuk URL request API...")

# Membentuk endpoint API e-Stat
url = f"""
http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData
?appId={APP_ID}
&lang=J
&statsDataId={STATS_DATA_ID}
&metaGetFlg=Y
&cntGetFlg=N
&explanationGetFlg=Y
&annotationGetFlg=Y
&sectionHeaderFlg=1
&replaceSpChars=0
"""

# Membersihkan format URL
url = url.replace("\n", "").replace(" ", "")

print("URL request berhasil dibuat.")

Mulai membentuk URL request API...
URL request berhasil dibuat.


In [4]:
# TAHAP 1D PENGAMBILAN DATA DARI API

print("Mulai mengambil data dari API e-Stat...")

# Mengirim request ke API
response = requests.get(url)

print("Request API selesai dilakukan.")
print(f"Status koneksi API : {response.status_code}")

Mulai mengambil data dari API e-Stat...
Request API selesai dilakukan.
Status koneksi API : 200


In [5]:
# TAHAP 1E PEMBACAAN STRUKTUR DATA RESPON API

print("Mulai membaca struktur data dari API...")

# Memisahkan isi response menjadi baris-baris teks
lines = response.text.splitlines()

# Mencari posisi awal tabel data asli
value_index = None

for i, line in enumerate(lines):
    if line.strip().replace('"', '') == "VALUE":
        value_index = i
        break

if value_index is not None:
    print("Penanda awal data berhasil ditemukan.")
    print(f"Posisi baris VALUE ditemukan pada index ke-{value_index}")
else:
    print("Penanda VALUE tidak ditemukan.")

Mulai membaca struktur data dari API...
Penanda awal data berhasil ditemukan.
Posisi baris VALUE ditemukan pada index ke-28


In [6]:
# TAHAP 1F MEMBACA DATA KE DATAFRAME

print("Mulai memuat data ke DataFrame...")

if value_index is not None:

    # Mengambil bagian isi tabel CSV
    csv_text = "\n".join(lines[value_index + 1:])

    # Membaca data ke DataFrame Pandas
    df_biaya_sewa_mentah = pd.read_csv(StringIO(csv_text))

    # --- TAMBAHAN PATH TARGET UNTUK DATA MENTAH BIAYA SEWA ---
    path_raw_biaya_sewa = "../data/raw/raw_biaya_sewa_japan.csv"

    # Mengekspor data mentah asli dari API ke folder raw
    df_biaya_sewa_mentah.to_csv(path_raw_biaya_sewa, index=False, encoding='utf-8')
    # ---------------------------------------------------------

    print("Data berhasil dimuat ke DataFrame dan disimpan fisik.")
    print(f"Lokasi penyimpanan mentah: {path_raw_biaya_sewa}")
else:
    print("Data gagal dimuat karena penanda VALUE tidak ditemukan.")

Mulai memuat data ke DataFrame...
Data berhasil dimuat ke DataFrame dan disimpan fisik.
Lokasi penyimpanan mentah: ../data/raw/raw_biaya_sewa_japan.csv


In [7]:
# TAHAP 1G VERIFIKASI DATA MENTAH

print("\n=============================================")
print("        TAHAP 1 SELESAI (BIAYA SEWA)         ")
print("=============================================")

print("Proses pengambilan data mentah berhasil dilakukan.")
print(f"Total data awal : {df_biaya_sewa_mentah.shape[0]} baris")
print(f"Total kolom data : {df_biaya_sewa_mentah.shape[1]} kolom")

print("\n--- Daftar Kolom Dataset ---")

print(df_biaya_sewa_mentah.columns.tolist())


        TAHAP 1 SELESAI (BIAYA SEWA)         
Proses pengambilan data mentah berhasil dilakukan.
Total data awal : 85104 baris
Total kolom data : 11 kolom

--- Daftar Kolom Dataset ---
['tab_code', '観測値', 'cat01_code', 'Ｌ\u3000家計', 'area_code', '地域', 'time_code', '調査年', 'unit', 'value', 'annotation']


In [8]:
# TAHAP 2A PERSIAPAN DATA FILTERING

print("Mulai menyiapkan data untuk proses filtering...")

# Membuat salinan data mentah
df_sewa_filter = df_biaya_sewa_mentah.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_sewa_filter.shape[0]} baris")

Mulai menyiapkan data untuk proses filtering...
Data berhasil disiapkan.
Total data awal : 85104 baris


In [9]:
# TAHAP 2B PENYARINGAN DATA PREFEKTUR

print("Mulai menyaring data prefektur...")

# Menghapus data nasional Jepang
df_sewa_filter = (
    df_sewa_filter[
        df_sewa_filter['地域'] != '全国'
    ]
)

print("Penyaringan prefektur berhasil dilakukan.")
print(f"Total data setelah filter prefektur : {df_sewa_filter.shape[0]} baris")

Mulai menyaring data prefektur...
Penyaringan prefektur berhasil dilakukan.
Total data setelah filter prefektur : 83331 baris


In [10]:
# TAHAP 2C PENYARINGAN INDIKATOR DATA

print("Mulai menyaring indikator biaya hidup dan persentase sewa...")

# Menentukan indikator yang digunakan
INDIKATOR_TARGET = [
    "#L02211",  # Total pengeluaran rumah tangga
    "#L02412"   # Persentase biaya tempat tinggal
]

# Menyaring data berdasarkan indikator
df_sewa_filter = (
    df_sewa_filter[
        df_sewa_filter['cat01_code'].isin(INDIKATOR_TARGET)
    ]
)

print("Penyaringan indikator berhasil dilakukan.")
print(f"Indikator terdeteksi : {df_sewa_filter['cat01_code'].unique().tolist()}")

Mulai menyaring indikator biaya hidup dan persentase sewa...
Penyaringan indikator berhasil dilakukan.
Indikator terdeteksi : ['#L02211', '#L02412']


In [11]:
# TAHAP 2D PENYARINGAN RENTANG TAHUN PENELITIAN

print("Mulai menyaring data berdasarkan tahun penelitian...")

# Menentukan rentang tahun penelitian
tahun_target_sewa = [
    '2020年度',
    '2021年度',
    '2022年度',
    '2023年度',
    '2024年度',
    '2025年度',
    '2026年度'
]

# Menyaring data berdasarkan tahun
df_sewa_filter = (
    df_sewa_filter[
        df_sewa_filter['調査年'].isin(tahun_target_sewa)
    ]
)

print("Penyaringan tahun berhasil dilakukan.")
print(f"Tahun terdeteksi : {df_sewa_filter['調査年'].unique().tolist()}")

Mulai menyaring data berdasarkan tahun penelitian...
Penyaringan tahun berhasil dilakukan.
Tahun terdeteksi : ['2020年度', '2021年度', '2022年度', '2023年度', '2024年度']


In [12]:
# TAHAP 2E VERIFIKASI HASIL FILTERING

print("\n=============================================")
print("         TAHAP 2 SELESAI (BIAYA SEWA)        ")
print("=============================================")

print("Penyaringan data biaya sewa berhasil dilakukan.")
print(f"Total data setelah filtering : {df_sewa_filter.shape[0]} baris")

print("\n--- Sampel Data Hasil Filtering ---")

kolom_tampil = [
    '地域',
    'cat01_code',
    '調査年',
    'unit',
    'value'
]

print(df_sewa_filter[kolom_tampil].head(5))


         TAHAP 2 SELESAI (BIAYA SEWA)        
Penyaringan data biaya sewa berhasil dilakukan.
Total data setelah filtering : 470 baris

--- Sampel Data Hasil Filtering ---
       地域 cat01_code     調査年 unit  value
8445  北海道    #L02211  2020年度   千円  301.7
8446  北海道    #L02211  2021年度   千円  268.4
8447  北海道    #L02211  2022年度   千円  277.7
8448  北海道    #L02211  2023年度   千円  296.9
8449  北海道    #L02211  2024年度   千円  285.5


In [13]:
# TAHAP 3A IMPORT LIBRARY NUMPY

print("Mulai mengimpor library NumPy...")

import numpy as np

print("Library NumPy berhasil diimpor.")

Mulai mengimpor library NumPy...
Library NumPy berhasil diimpor.


In [14]:
# TAHAP 3B PERSIAPAN DATA PEMBERSIHAN

print("Mulai menyiapkan data untuk proses pembersihan...")

# Membuat salinan data hasil filtering
df_sewa_clean = df_sewa_filter.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_sewa_clean.shape[0]} baris")

Mulai menyiapkan data untuk proses pembersihan...
Data berhasil disiapkan.
Total data awal : 470 baris


In [15]:
# TAHAP 3C PEMBERSIHAN NILAI KOSONG

print("Mulai membersihkan nilai kosong pada kolom value...")

# Menghapus spasi tersembunyi
df_sewa_clean['value'] = (
    df_sewa_clean['value']
    .astype(str)
    .str.strip()
)

# Mengganti simbol kosong menjadi NaN
df_sewa_clean['value'] = (
    df_sewa_clean['value']
    .replace(['-', '‐', ''], np.nan)
)

print("Pembersihan nilai kosong berhasil dilakukan.")

Mulai membersihkan nilai kosong pada kolom value...
Pembersihan nilai kosong berhasil dilakukan.


In [16]:
# TAHAP 3D PENGHAPUSAN DATA NULL

print("Mulai menghapus data yang memiliki nilai kosong...")

# Menghapus baris dengan nilai kosong pada kolom value
df_sewa_clean = (
    df_sewa_clean
    .dropna(subset=['value'])
)

print("Data kosong berhasil dihapus.")
print(f"Total data setelah penghapusan null : {df_sewa_clean.shape[0]} baris")

Mulai menghapus data yang memiliki nilai kosong...
Data kosong berhasil dihapus.
Total data setelah penghapusan null : 470 baris


In [17]:
# TAHAP 3E KONVERSI TIPE DATA

print("Mulai mengubah tipe data value menjadi float...")

# Mengubah tipe data menjadi float
df_sewa_clean['value'] = (
    df_sewa_clean['value']
    .astype(float)
)

print("Konversi tipe data berhasil dilakukan.")

Mulai mengubah tipe data value menjadi float...
Konversi tipe data berhasil dilakukan.


In [18]:
# TAHAP 3F VERIFIKASI HASIL PEMBERSIHAN DATA

print("\n=============================================")
print("         TAHAP 3 SELESAI (BIAYA SEWA)        ")
print("=============================================")

print("Pembersihan data dan konversi tipe data berhasil dilakukan.")
print(f"Total data saat ini : {df_sewa_clean.shape[0]} baris")

print("\n--- Verifikasi Struktur Kolom Value ---")

print(df_sewa_clean[['value']].info())


         TAHAP 3 SELESAI (BIAYA SEWA)        
Pembersihan data dan konversi tipe data berhasil dilakukan.
Total data saat ini : 470 baris

--- Verifikasi Struktur Kolom Value ---
<class 'pandas.core.frame.DataFrame'>
Index: 470 entries, 8445 to 11999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   value   470 non-null    float64
dtypes: float64(1)
memory usage: 7.3 KB
None


In [19]:
# TAHAP 4A PERSIAPAN STRUKTURISASI DATA

print("Mulai menyiapkan proses strukturisasi data...")

# Membuat salinan data hasil pembersihan
df_sewa_pivot = df_sewa_clean.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_sewa_pivot.shape[0]} baris")

Mulai menyiapkan proses strukturisasi data...
Data berhasil disiapkan.
Total data awal : 470 baris


In [20]:
# TAHAP 4B TRANSFORMASI DATA DENGAN PIVOT

print("Mulai memisahkan indikator ke dalam kolom terpisah...")

# Mengubah indikator menjadi kolom terpisah
df_sewa_pivot = (
    df_sewa_pivot
    .pivot(
        index=['地域', '調査年'],
        columns='cat01_code',
        values='value'
    )
    .reset_index()
)

print("Transformasi pivot berhasil dilakukan.")

Mulai memisahkan indikator ke dalam kolom terpisah...
Transformasi pivot berhasil dilakukan.


In [21]:
# TAHAP 4C PENAMAAN ULANG KOLOM HASIL PIVOT

print("Mulai melakukan penyesuaian nama kolom...")

# Mengubah nama kolom agar lebih mudah dipahami
df_sewa_pivot = df_sewa_pivot.rename(columns={
    '#L02211': 'biaya_hidup_mentah',
    '#L02412': 'persentase_sewa'
})

print("Penyesuaian nama kolom berhasil dilakukan.")

Mulai melakukan penyesuaian nama kolom...
Penyesuaian nama kolom berhasil dilakukan.


In [22]:
# TAHAP 4D KONVERSI NILAI BIAYA HIDUP KE YEN ASLI

print("Mulai menghitung nominal biaya hidup dalam Yen...")

# Mengubah satuan ribuan Yen menjadi Yen asli
df_sewa_pivot['biaya_hidup_nominal_yen'] = (
    df_sewa_pivot['biaya_hidup_mentah'] * 1000
)

print("Konversi nominal biaya hidup berhasil dilakukan.")

Mulai menghitung nominal biaya hidup dalam Yen...
Konversi nominal biaya hidup berhasil dilakukan.


In [23]:
# TAHAP 4E PERHITUNGAN NOMINAL BIAYA SEWA

print("Mulai menghitung nominal biaya sewa...")

# Menghitung nominal biaya sewa berdasarkan persentase tempat tinggal
df_sewa_pivot['biaya_sewa_nominal_yen'] = (
    (
        df_sewa_pivot['persentase_sewa'] / 100
    ) * df_sewa_pivot['biaya_hidup_nominal_yen']
)

print("Perhitungan nominal biaya sewa berhasil dilakukan.")

Mulai menghitung nominal biaya sewa...
Perhitungan nominal biaya sewa berhasil dilakukan.


In [24]:
# TAHAP 4F PEMBULATAN HASIL PERHITUNGAN

print("Mulai membulatkan hasil nominal biaya sewa...")

# Membulatkan hasil menjadi integer
df_sewa_pivot['biaya_sewa_nominal_yen'] = (
    df_sewa_pivot['biaya_sewa_nominal_yen']
    .round()
    .astype('int64')
)

print("Pembulatan nominal biaya sewa berhasil dilakukan.")

Mulai membulatkan hasil nominal biaya sewa...
Pembulatan nominal biaya sewa berhasil dilakukan.


In [25]:
# TAHAP 4G VERIFIKASI HASIL PERHITUNGAN

print("\n=============================================")
print("         TAHAP 4 SELESAI (BIAYA SEWA)        ")
print("=============================================")

print("Strukturisasi data dan perhitungan biaya sewa berhasil dilakukan.")
print(f"Total data hasil pivot : {df_sewa_pivot.shape[0]} baris")

print("\n--- Sampel Hasil Perhitungan Nominal Sewa ---")

kolom_tampil = [
    '地域',
    '調査年',
    'persentase_sewa',
    'biaya_hidup_nominal_yen',
    'biaya_sewa_nominal_yen'
]

print(df_sewa_pivot[kolom_tampil].head(5))


         TAHAP 4 SELESAI (BIAYA SEWA)        
Strukturisasi data dan perhitungan biaya sewa berhasil dilakukan.
Total data hasil pivot : 235 baris

--- Sampel Hasil Perhitungan Nominal Sewa ---
cat01_code   地域     調査年  persentase_sewa  biaya_hidup_nominal_yen  \
0           三重県  2020年度              5.4                 285100.0   
1           三重県  2021年度              5.2                 295800.0   
2           三重県  2022年度              5.6                 277100.0   
3           三重県  2023年度              6.1                 332700.0   
4           三重県  2024年度              8.8                 289400.0   

cat01_code  biaya_sewa_nominal_yen  
0                            15395  
1                            15382  
2                            15518  
3                            20295  
4                            25467  


In [26]:
# TAHAP 5A PERSIAPAN DATA TRANSLASI

print("Mulai menyiapkan data untuk proses translasi...")

# Membuat salinan data hasil perhitungan nominal
df_sewa_translate = df_sewa_pivot.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_sewa_translate.shape[0]} baris")

Mulai menyiapkan data untuk proses translasi...
Data berhasil disiapkan.
Total data awal : 235 baris


In [27]:
# TAHAP 5B PEMBUATAN KAMUS TRANSLASI PREFEKTUR

print("Mulai membuat kamus translasi prefektur Jepang...")

# Kamus translasi prefektur Jepang
dict_prefektur_sewa = {
    '北海道': 'Hokkaido',
    '青森県': 'Aomori',
    '岩手県': 'Iwate',
    '宮城県': 'Miyagi',
    '秋田県': 'Akita',
    '山形県': 'Yamagata',
    '福島県': 'Fukushima',
    '茨城県': 'Ibaraki',
    '栃木県': 'Tochigi',
    '群馬県': 'Gunma',
    '埼玉県': 'Saitama',
    '千葉県': 'Chiba',
    '東京都': 'Tokyo',
    '神奈川県': 'Kanagawa',
    '新潟県': 'Niigata',
    '富山県': 'Toyama',
    '石川県': 'Ishikawa',
    '福井県': 'Fukui',
    '山梨県': 'Yamanashi',
    '長野県': 'Nagano',
    '岐阜県': 'Gifu',
    '静岡県': 'Shizuoka',
    '愛知県': 'Aichi',
    '三重県': 'Mie',
    '滋賀県': 'Shiga',
    '京都府': 'Kyoto',
    '大阪府': 'Osaka',
    '兵庫県': 'Hyogo',
    '奈良県': 'Nara',
    '和歌山県': 'Wakayama',
    '鳥取県': 'Tottori',
    '島根県': 'Shimane',
    '岡山県': 'Okayama',
    '広島県': 'Hiroshima',
    '山口県': 'Yamaguchi',
    '徳島県': 'Tokushima',
    '香川県': 'Kagawa',
    '愛媛県': 'Ehime',
    '高知県': 'Kochi',
    '福岡県': 'Fukuoka',
    '佐賀県': 'Saga',
    '長崎県': 'Nagasaki',
    '熊本県': 'Kumamoto',
    '大分県': 'Oita',
    '宮崎県': 'Miyazaki',
    '鹿児島県': 'Kagoshima',
    '沖縄県': 'Okinawa'
}

print("Kamus translasi berhasil dibuat.")

Mulai membuat kamus translasi prefektur Jepang...
Kamus translasi berhasil dibuat.


In [28]:
# TAHAP 5C TRANSLASI NAMA PREFEKTUR

print("Mulai menerjemahkan nama prefektur ke Bahasa Inggris...")

# Translasi nama prefektur Jepang
df_sewa_translate['prefektur_en'] = (
    df_sewa_translate['地域']
    .map(dict_prefektur_sewa)
)

print("Translasi prefektur berhasil dilakukan.")

Mulai menerjemahkan nama prefektur ke Bahasa Inggris...
Translasi prefektur berhasil dilakukan.


In [29]:
# TAHAP 5D PENAMBAHAN LABEL KATEGORI BIAYA

print("Mulai menambahkan label kategori biaya...")

# Menambahkan kategori biaya dalam Bahasa Indonesia
df_sewa_translate['kategori_biaya_id'] = 'Biaya Sewa Bulanan'

print("Kategori biaya berhasil ditambahkan.")

Mulai menambahkan label kategori biaya...
Kategori biaya berhasil ditambahkan.


In [30]:
# TAHAP 5E VERIFIKASI HASIL TRANSLASI

print("\n=============================================")
print("        TAHAP 5 SELESAI (BIAYA SEWA)         ")
print("=============================================")

print("Penambahan kolom translasi berhasil dilakukan.")
print(f"Total data saat ini : {df_sewa_translate.shape[0]} baris")

print("\n--- Sampel Hasil Translasi Biaya Sewa ---")

kolom_tampil = [
    '地域',
    'prefektur_en',
    '調査年',
    'kategori_biaya_id',
    'biaya_sewa_nominal_yen'
]

print(df_sewa_translate[kolom_tampil].head(5))


        TAHAP 5 SELESAI (BIAYA SEWA)         
Penambahan kolom translasi berhasil dilakukan.
Total data saat ini : 235 baris

--- Sampel Hasil Translasi Biaya Sewa ---
cat01_code   地域 prefektur_en     調査年   kategori_biaya_id  \
0           三重県          Mie  2020年度  Biaya Sewa Bulanan   
1           三重県          Mie  2021年度  Biaya Sewa Bulanan   
2           三重県          Mie  2022年度  Biaya Sewa Bulanan   
3           三重県          Mie  2023年度  Biaya Sewa Bulanan   
4           三重県          Mie  2024年度  Biaya Sewa Bulanan   

cat01_code  biaya_sewa_nominal_yen  
0                            15395  
1                            15382  
2                            15518  
3                            20295  
4                            25467  


In [31]:
# TAHAP 6A PERSIAPAN DATA ATRIBUT SAW

print("Mulai menyiapkan data atribut SAW...")

# Membuat salinan data hasil translasi
df_sewa_saw = df_sewa_translate.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_sewa_saw.shape[0]} baris")

Mulai menyiapkan data atribut SAW...
Data berhasil disiapkan.
Total data awal : 235 baris


In [32]:
# TAHAP 6B PENAMBAHAN ATRIBUT KRITERIA SAW

print("Mulai menambahkan atribut tipe kriteria SAW...")

# Menambahkan atribut Cost untuk biaya sewa
df_sewa_saw['tipe_kriteria_sewa'] = 'Cost'

print("Atribut tipe kriteria berhasil ditambahkan.")

Mulai menambahkan atribut tipe kriteria SAW...
Atribut tipe kriteria berhasil ditambahkan.


In [33]:
# TAHAP 6C VERIFIKASI HASIL ATRIBUT SAW

print("\n=============================================")
print("        TAHAP 6 SELESAI (BIAYA SEWA)         ")
print("=============================================")

print("Penambahan atribut SAW berhasil dilakukan.")
print(f"Total data saat ini : {df_sewa_saw.shape[0]} baris")

print("\n--- Sampel Data Hasil Atribut SAW ---")

kolom_tampil = [
    'prefektur_en',
    '調査年',
    'biaya_sewa_nominal_yen',
    'tipe_kriteria_sewa'
]

print(df_sewa_saw[kolom_tampil].head(5))


        TAHAP 6 SELESAI (BIAYA SEWA)         
Penambahan atribut SAW berhasil dilakukan.
Total data saat ini : 235 baris

--- Sampel Data Hasil Atribut SAW ---
cat01_code prefektur_en     調査年  biaya_sewa_nominal_yen tipe_kriteria_sewa
0                   Mie  2020年度                   15395               Cost
1                   Mie  2021年度                   15382               Cost
2                   Mie  2022年度                   15518               Cost
3                   Mie  2023年度                   20295               Cost
4                   Mie  2024年度                   25467               Cost


In [34]:
# TAHAP 7A PERSIAPAN AGREGASI DATA

print("Mulai menyiapkan proses agregasi data biaya sewa...")

# Membuat salinan data hasil atribut SAW
df_sewa_final = df_sewa_saw.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_sewa_final.shape[0]} baris")

Mulai menyiapkan proses agregasi data biaya sewa...
Data berhasil disiapkan.
Total data awal : 235 baris


In [35]:
# TAHAP 7B AGREGASI RATA-RATA BIAYA SEWA

print("Mulai menghitung rata-rata biaya sewa per prefektur...")

# Menghitung rata-rata biaya sewa
df_sewa_final = (
    df_sewa_final
    .groupby(
        [
            'prefektur_en',
            'kategori_biaya_id',
            'tipe_kriteria_sewa'
        ],
        as_index=False
    )
    .agg({
        'biaya_sewa_nominal_yen': 'mean'
    })
)

print("Agregasi rata-rata biaya sewa berhasil dilakukan.")

Mulai menghitung rata-rata biaya sewa per prefektur...
Agregasi rata-rata biaya sewa berhasil dilakukan.


In [36]:
# TAHAP 7C PEMBULATAN HASIL AGREGASI

print("Mulai membulatkan hasil agregasi...")

# Membulatkan nilai rata-rata biaya sewa
df_sewa_final['biaya_sewa_nominal_yen'] = (
    df_sewa_final['biaya_sewa_nominal_yen']
    .round()
    .astype('int64')
)

print("Pembulatan hasil agregasi berhasil dilakukan.")

Mulai membulatkan hasil agregasi...
Pembulatan hasil agregasi berhasil dilakukan.


In [37]:
# TAHAP 7D PENGURUTAN DATA FINAL

print("Mulai mengurutkan data berdasarkan nama prefektur...")

# Mengurutkan data secara alfabetis
df_sewa_final = (
    df_sewa_final
    .sort_values(by=['prefektur_en'])
    .reset_index(drop=True)
)

print("Pengurutan data berhasil dilakukan.")

Mulai mengurutkan data berdasarkan nama prefektur...
Pengurutan data berhasil dilakukan.


In [38]:
# TAHAP 7E EKSPOR DATASET FINAL

print("Mulai mengekspor dataset final biaya sewa...")

# --- PERUBAHAN PATH TARGET UNTUK DATA FINAL BIAYA SEWA ---
nama_file_csv_sewa = "../data/processed/dataset_biaya_sewa_final.csv"

# Mengekspor dataset ke CSV
df_sewa_final.to_csv(
    nama_file_csv_sewa,
    index=False,
    encoding='utf-8'
)

print("Ekspor dataset berhasil dilakukan.")
print(f"Nama file / Jalur : {nama_file_csv_sewa}")

Mulai mengekspor dataset final biaya sewa...
Ekspor dataset berhasil dilakukan.
Nama file / Jalur : ../data/processed/dataset_biaya_sewa_final.csv


In [39]:
# TAHAP 7F VERIFIKASI HASIL DATASET FINAL

print("\n=============================================")
print("        TAHAP 7 SELESAI (BIAYA SEWA)         ")
print("=============================================")

print("Agregasi dan ekspor dataset biaya sewa berhasil dilakukan.")
print(f"Total data final : {df_sewa_final.shape[0]} baris")

print("\n--- Sampel Dataset Final Biaya Sewa ---")

print(df_sewa_final.head(5))


        TAHAP 7 SELESAI (BIAYA SEWA)         
Agregasi dan ekspor dataset biaya sewa berhasil dilakukan.
Total data final : 47 baris

--- Sampel Dataset Final Biaya Sewa ---
cat01_code prefektur_en   kategori_biaya_id tipe_kriteria_sewa  \
0                 Aichi  Biaya Sewa Bulanan               Cost   
1                 Akita  Biaya Sewa Bulanan               Cost   
2                Aomori  Biaya Sewa Bulanan               Cost   
3                 Chiba  Biaya Sewa Bulanan               Cost   
4                 Ehime  Biaya Sewa Bulanan               Cost   

cat01_code  biaya_sewa_nominal_yen  
0                            16645  
1                            12628  
2                            13927  
3                            20118  
4                            15187  


In [40]:
print(df_sewa_final.columns.tolist())

['prefektur_en', 'kategori_biaya_id', 'tipe_kriteria_sewa', 'biaya_sewa_nominal_yen']
